In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta

In [3]:
lake_lse_gdf_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision.pkl'
lake_lse_gdf_save_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
lake_lse_gdf = pd.read_pickle(lake_lse_gdf_path)
lake_lse_gdf = lake_lse_gdf.set_index('Hylak_id')

hybas_id_list = [
    1020000010, 1020011530, 1020018110, 1020021940, 1020027430, 1020034170, 1020035180, 1020040190,
    2020000010, 2020003440, 2020018240, 2020024230, 2020033490, 2020041390, 2020057170, 2020065840, 2020071190,
    3020000010, 3020003790, 3020005240, 3020008670, 3020009320, 3020024310,
    4020000010, 4020006940, 4020015090, 4020024190, 4020034510, 4020050210, 4020050220, 4020050290, 4020050470,
    5020000010, 5020015660, 5020037270, 5020049720, 5020082270, 
    6020000010, 6020006540, 6020008320, 6020014330, 6020017370, 6020021870, 6020029280,
    7020000010, 7020014250, 7020021430, 7020024600, 7020038340, 7020046750, 7020047840, 7020065090,
    8020000010, 8020008900, 8020010700, 8020020760, 8020022890, 8020032840, 8020044560,
    9020000010
]
gsw_missing_area_csv_path_pattern = '/WORK/Data/global_lake_area/area_csvs/gsw_missing_data_area_concatenated/{basin_id}_gsw_missing_data_area_concatenated.csv'
my_missing_area_csv_path_pattern = '/WORK/Data/global_lake_area/area_csvs/missing_data_area_concatenated/{BASIN_ID}_missing_data_area_concatenated.csv'

common_area_columns = []
start_date = '2001-01-01'
end_date = '2022-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns.append(current_date_str)
    current_date += relativedelta(months=1)
gsw_missing_area_columns_rename_dict = {col: f'gsw_missing_{col}' for col in common_area_columns}
my_missing_area_columns_rename_dict = {col: f'my_missing_{col}' for col in common_area_columns}
gsw_missing_area_columns = list(gsw_missing_area_columns_rename_dict.values())
my_missing_area_columns = list(my_missing_area_columns_rename_dict.values())

lake_lse_gdf[gsw_missing_area_columns] = np.nan
lake_lse_gdf[my_missing_area_columns] = np.nan

lake_boundary_in_ea_proj_path_pattern = '/WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_buffered/hylak_buffered_updated_no_contained_{basin_id}_reprojected.shp'

lake_lse_gdf['lake_boundary_area'] = np.nan

for hybas_id in hybas_id_list:
    print(f'Processing basin {hybas_id}')
    gsw_missing_area_csv_path = gsw_missing_area_csv_path_pattern.format(basin_id=hybas_id)
    my_missing_area_csv_path = my_missing_area_csv_path_pattern.format(BASIN_ID=hybas_id)
    lake_boundary_in_ea_proj_path = lake_boundary_in_ea_proj_path_pattern.format(basin_id=hybas_id)
    lake_boundary_in_ea_proj_gdf = gpd.read_file(lake_boundary_in_ea_proj_path).set_index('Hylak_id')
    lake_boundary_in_ea_proj_gdf['lake_boundary_area'] = lake_boundary_in_ea_proj_gdf['geometry'].area
    lake_lse_gdf.update(lake_boundary_in_ea_proj_gdf['lake_boundary_area'])
    gsw_missing_area_df = pd.read_csv(gsw_missing_area_csv_path).set_index('Hylak_id')
    gsw_missing_area_df = gsw_missing_area_df.rename(columns=gsw_missing_area_columns_rename_dict)
    my_missing_area_df = pd.read_csv(my_missing_area_csv_path).set_index('Hylak_id')
    my_missing_area_df = my_missing_area_df.rename(columns=my_missing_area_columns_rename_dict)
    lake_lse_gdf.update(gsw_missing_area_df[gsw_missing_area_columns])
    lake_lse_gdf.update(my_missing_area_df[my_missing_area_columns])

lake_lse_gdf.to_pickle(lake_lse_gdf_save_path)

In [5]:
lake_lse_gdf['geometry']

Hylak_id
2          MULTIPOLYGON (((-118.03450 65.89200, -118.0345...
3          MULTIPOLYGON (((-112.06350 61.90375, -112.0635...
4          MULTIPOLYGON (((-97.68451 53.48093, -97.68451 ...
5          MULTIPOLYGON (((-92.00626 46.68239, -92.00653 ...
6          MULTIPOLYGON (((-87.90375 43.01425, -87.90375 ...
                                 ...                        
1427543    POLYGON ((167.89875 -46.74725, 167.89875 -46.7...
1427684    POLYGON ((169.14763 -52.59291, 169.14750 -52.5...
1427686    POLYGON ((158.89124 -54.53117, 158.88696 -54.5...
1427687    POLYGON ((158.88858 -54.59767, 158.88833 -54.5...
1427688    POLYGON ((158.83901 -54.68744, 158.83867 -54.6...
Name: geometry, Length: 1401802, dtype: geometry

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta

lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
lake_lse_gdf_with_missing_data_ratio_save_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_ratio.pkl'
lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path)

common_area_columns = []
start_date = '2001-01-01'
end_date = '2022-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns.append(current_date_str)
    current_date += relativedelta(months=1)
gsw_missing_area_columns_rename_dict = {col: f'gsw_missing_{col}' for col in common_area_columns}
my_missing_area_columns_rename_dict = {col: f'my_missing_{col}' for col in common_area_columns}
gsw_missing_area_columns = list(gsw_missing_area_columns_rename_dict.values())
my_missing_area_columns = list(my_missing_area_columns_rename_dict.values())

geometry_area = lake_lse_gdf_with_missing_data['lake_boundary_area']
for gsw_missing_area_column in gsw_missing_area_columns:
    lake_lse_gdf_with_missing_data[gsw_missing_area_column] = lake_lse_gdf_with_missing_data[gsw_missing_area_column] / geometry_area
for my_missing_area_column in my_missing_area_columns:
    lake_lse_gdf_with_missing_data[my_missing_area_column] = lake_lse_gdf_with_missing_data[my_missing_area_column] / geometry_area

lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data.reset_index()

lake_lse_gdf_with_missing_data.to_pickle(lake_lse_gdf_with_missing_data_ratio_save_path)

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics.pkl'
hydrobasins_gdf_with_missing_data_count_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_path)

start_date = '2001-01-01'
end_date = '2012-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_first_half = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_first_half.append(current_date_str)
    current_date += relativedelta(months=1)

start_date = '2012-01-01'
end_date = '2022-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_second_half = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_second_half.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_first_half, common_area_columns_second_half]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_ratio_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_ratio.pkl'
    lake_lse_gdf_with_missing_data_ratio = pd.read_pickle(lake_lse_gdf_with_missing_data_ratio_path)

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data_ratio.crs



    lake_lse_gdf_with_missing_data_ratio = lake_lse_gdf_with_missing_data_ratio[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]

    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'] > cloud_threshold
        lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'] > cloud_threshold
        lake_lse_gdf_with_missing_data_ratio[f'lake_count_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio['lake_boundary_area'].notna()
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'] = (lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'].astype(bool)) & (~lake_lse_gdf_with_missing_data_ratio[f'frozen_{common_area_column}'].astype(bool))
            lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'] = (lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'].astype(bool)) & (~lake_lse_gdf_with_missing_data_ratio[f'frozen_{common_area_column}'].astype(bool))
            lake_lse_gdf_with_missing_data_ratio[f'lake_count_{common_area_column}'] = (lake_lse_gdf_with_missing_data_ratio[f'lake_count_{common_area_column}'].astype(bool)) & (~lake_lse_gdf_with_missing_data_ratio[f'frozen_{common_area_column}'].astype(bool))
            lake_lse_gdf_with_missing_data_ratio.drop(columns=[f'frozen_{common_area_column}'], inplace=True)
        lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio[f'gsw_missing_{common_area_column}'].astype(int)
        lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio[f'my_missing_{common_area_column}'].astype(int)
        lake_lse_gdf_with_missing_data_ratio[f'lake_count_{common_area_column}'] = lake_lse_gdf_with_missing_data_ratio[f'lake_count_{common_area_column}'].astype(int)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data_ratio, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_counts = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_counts = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        # rename to add "count"
        gsw_missing_counts = gsw_missing_counts.rename(f'gsw_missing_count_{common_area_column}')
        my_missing_counts = my_missing_counts.rename(f'my_missing_count_{common_area_column}')
        lake_counts = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_count_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_counts, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_counts, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_counts, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2001-01-01
Processing 2001-02-01
Processing 2001-03-01
Processing 2001-04-01
Processing 2001-05-01
Processing 2001-06-01
Processing 2001-07-01
Processing 2001-08-01
Processing 2001-09-01
Processing 2001-10-01
Processing 2001-11-01
Processing 2001-12-01
Processing 2002-01-01
Processing 2002-02-01
Processing 2002-03-01
Processing 2002-04-01
Processing 2002-05-01
Processing 2002-06-01
Processing 2002-07-01
Processing 2002-08-01
Processing 2002-09-01
Processing 2002-10-01
Processing 2002-11-01
Processing 2002-12-01
Processing 2003-01-01
Processing 2003-02-01
Processing 2003-03-01
Processing 2003-04-01
Processing 2003-05-01
Processing 2003-06-01
Processing 2003-07-01
Processing 2003-08-01
Processing 2003-09-01
Processing 2003-10-01
Processing 2003-11-01
Processing 2003-12-01
Processing 2004-01-01
Processing 2004-02-01
Processing 2004-03-01
Processing 2004-04-01
Processing 2004-05-01
Processing 2004-06-01
Processing 2004-07-01
Processing 2004-08-01
Processing 2004-09-01
Processing

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2012-01-01
Processing 2012-02-01
Processing 2012-03-01
Processing 2012-04-01
Processing 2012-05-01
Processing 2012-06-01
Processing 2012-07-01
Processing 2012-08-01
Processing 2012-09-01
Processing 2012-10-01
Processing 2012-11-01
Processing 2012-12-01
Processing 2013-01-01
Processing 2013-02-01
Processing 2013-03-01
Processing 2013-04-01
Processing 2013-05-01
Processing 2013-06-01
Processing 2013-07-01
Processing 2013-08-01
Processing 2013-09-01
Processing 2013-10-01
Processing 2013-11-01
Processing 2013-12-01
Processing 2014-01-01
Processing 2014-02-01
Processing 2014-03-01
Processing 2014-04-01
Processing 2014-05-01
Processing 2014-06-01
Processing 2014-07-01
Processing 2014-08-01
Processing 2014-09-01
Processing 2014-10-01
Processing 2014-11-01
Processing 2014-12-01
Processing 2015-01-01
Processing 2015-02-01
Processing 2015-03-01
Processing 2015-04-01
Processing 2015-05-01
Processing 2015-06-01
Processing 2015-07-01
Processing 2015-08-01
Processing 2015-09-01
Processing

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_first_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_path)

start_date = '2001-01-01'
end_date = '2006-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_first_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_first_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_first_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2001-01-01
Processing 2001-02-01
Processing 2001-03-01
Processing 2001-04-01
Processing 2001-05-01
Processing 2001-06-01
Processing 2001-07-01
Processing 2001-08-01
Processing 2001-09-01
Processing 2001-10-01
Processing 2001-11-01
Processing 2001-12-01
Processing 2002-01-01
Processing 2002-02-01
Processing 2002-03-01
Processing 2002-04-01
Processing 2002-05-01
Processing 2002-06-01
Processing 2002-07-01
Processing 2002-08-01
Processing 2002-09-01
Processing 2002-10-01
Processing 2002-11-01
Processing 2002-12-01
Processing 2003-01-01
Processing 2003-02-01
Processing 2003-03-01
Processing 2003-04-01
Processing 2003-05-01
Processing 2003-06-01
Processing 2003-07-01
Processing 2003-08-01
Processing 2003-09-01
Processing 2003-10-01
Processing 2003-11-01
Processing 2003-12-01
Processing 2004-01-01
Processing 2004-02-01
Processing 2004-03-01
Processing 2004-04-01
Processing 2004-05-01
Processing 2004-06-01
Processing 2004-07-01
Processing 2004-08-01
Processing 2004-09-01
Processing

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_first_quarter.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_second_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path)

start_date = '2006-01-01'
end_date = '2011-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_second_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_second_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_second_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2006-01-01
Processing 2006-02-01
Processing 2006-03-01
Processing 2006-04-01
Processing 2006-05-01
Processing 2006-06-01
Processing 2006-07-01
Processing 2006-08-01
Processing 2006-09-01
Processing 2006-10-01
Processing 2006-11-01
Processing 2006-12-01
Processing 2007-01-01
Processing 2007-02-01
Processing 2007-03-01
Processing 2007-04-01
Processing 2007-05-01
Processing 2007-06-01
Processing 2007-07-01
Processing 2007-08-01
Processing 2007-09-01
Processing 2007-10-01
Processing 2007-11-01
Processing 2007-12-01
Processing 2008-01-01
Processing 2008-02-01
Processing 2008-03-01
Processing 2008-04-01
Processing 2008-05-01
Processing 2008-06-01
Processing 2008-07-01
Processing 2008-08-01
Processing 2008-09-01
Processing 2008-10-01
Processing 2008-11-01
Processing 2008-12-01
Processing 2009-01-01
Processing 2009-02-01
Processing 2009-03-01
Processing 2009-04-01
Processing 2009-05-01
Processing 2009-06-01
Processing 2009-07-01
Processing 2009-08-01
Processing 2009-09-01
Processing

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_second_quarter.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_third_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path)

start_date = '2011-01-01'
end_date = '2014-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_third_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_third_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_third_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2011-01-01
Processing 2011-02-01
Processing 2011-03-01
Processing 2011-04-01
Processing 2011-05-01
Processing 2011-06-01
Processing 2011-07-01
Processing 2011-08-01
Processing 2011-09-01
Processing 2011-10-01
Processing 2011-11-01
Processing 2011-12-01
Processing 2012-01-01
Processing 2012-02-01
Processing 2012-03-01
Processing 2012-04-01
Processing 2012-05-01
Processing 2012-06-01
Processing 2012-07-01
Processing 2012-08-01
Processing 2012-09-01
Processing 2012-10-01
Processing 2012-11-01
Processing 2012-12-01
Processing 2013-01-01
Processing 2013-02-01
Processing 2013-03-01
Processing 2013-04-01
Processing 2013-05-01
Processing 2013-06-01
Processing 2013-07-01
Processing 2013-08-01
Processing 2013-09-01
Processing 2013-10-01
Processing 2013-11-01
Processing 2013-12-01


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_third_quarter.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_forth_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path)

start_date = '2014-01-01'
end_date = '2016-06-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_forth_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_forth_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_forth_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


Processing 2014-01-01
Processing 2014-02-01
Processing 2014-03-01
Processing 2014-04-01
Processing 2014-05-01
Processing 2014-06-01
Processing 2014-07-01
Processing 2014-08-01
Processing 2014-09-01
Processing 2014-10-01
Processing 2014-11-01
Processing 2014-12-01
Processing 2015-01-01
Processing 2015-02-01
Processing 2015-03-01
Processing 2015-04-01
Processing 2015-05-01
Processing 2015-06-01
Processing 2015-07-01
Processing 2015-08-01
Processing 2015-09-01
Processing 2015-10-01
Processing 2015-11-01
Processing 2015-12-01
Processing 2016-01-01
Processing 2016-02-01
Processing 2016-03-01
Processing 2016-04-01
Processing 2016-05-01


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_forth_quarter.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_fifth_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path)

start_date = '2016-06-01'
end_date = '2019-06-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_fifth_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_fifth_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_fifth_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Processing 2016-06-01
Processing 2016-07-01
Processing 2016-08-01
Processing 2016-09-01
Processing 2016-10-01
Processing 2016-11-01
Processing 2016-12-01
Processing 2017-01-01
Processing 2017-02-01
Processing 2017-03-01
Processing 2017-04-01
Processing 2017-05-01
Processing 2017-06-01
Processing 2017-07-01
Processing 2017-08-01
Processing 2017-09-01
Processing 2017-10-01
Processing 2017-11-01
Processing 2017-12-01
Processing 2018-01-01
Processing 2018-02-01
Processing 2018-03-01
Processing 2018-04-01
Processing 2018-05-01
Processing 2018-06-01
Processing 2018-07-01
Processing 2018-08-01
Processing 2018-09-01
Processing 2018-10-01
Processing 2018-11-01
Processing 2018-12-01
Processing 2019-01-01
Processing 2019-02-01
Processing 2019-03-01
Processing 2019-04-01
Processing 2019-05-01


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_fifth_quarter.pkl'
hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_full_quarter.pkl'
hydrobasins_gdf = pd.read_pickle(hydrobasins_gdf_with_missing_data_count_and_area_last_quarter_path)

start_date = '2019-06-01'
end_date = '2022-01-01'
date_fmt = '%Y-%m-%d'
start_date = datetime.strptime(start_date, date_fmt)
end_date = datetime.strptime(end_date, date_fmt)
common_area_columns_sixth_quarter = []
current_date = start_date
while current_date < end_date:
    current_date_str = current_date.strftime(date_fmt)
    common_area_columns_sixth_quarter.append(current_date_str)
    current_date += relativedelta(months=1)

cloud_threshold = 0.05


for common_area_columns in [common_area_columns_sixth_quarter]:
    gsw_missing_area_ratio_columns = [f'gsw_missing_{col}' for col in common_area_columns]
    my_missing_area_ratio_columns = [f'my_missing_{col}' for col in common_area_columns]
    frozen_flag_columns = [f'frozen_{col}' for col in common_area_columns]
    
    lake_lse_gdf_with_missing_data_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
    lake_lse_gdf_with_missing_data = pd.read_pickle(lake_lse_gdf_with_missing_data_path).reset_index()

    assert hydrobasins_gdf.crs == lake_lse_gdf_with_missing_data.crs



    lake_lse_gdf_with_missing_data = lake_lse_gdf_with_missing_data[['Hylak_id', 'centroid', 'lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns + frozen_flag_columns]
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns]/1e6
    lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns] = lake_lse_gdf_with_missing_data[['lake_boundary_area'] + gsw_missing_area_ratio_columns + my_missing_area_ratio_columns].astype('float32')
    EXCLUDE_FROZEN = True

    for common_area_column in common_area_columns:
        lake_lse_gdf_with_missing_data[f'lake_total_boundary_area_{common_area_column}'] = lake_lse_gdf_with_missing_data['lake_boundary_area']
        if EXCLUDE_FROZEN:
            lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'] = lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'].astype(bool)
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'gsw_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'my_missing_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.loc[lake_lse_gdf_with_missing_data[f'frozen_{common_area_column}'], f'lake_total_boundary_area_{common_area_column}'] = 0
            lake_lse_gdf_with_missing_data.drop(columns=[f'frozen_{common_area_column}'], inplace=True)

    hydrobasins_with_lakes_gdf = gpd.sjoin(hydrobasins_gdf, lake_lse_gdf_with_missing_data, how='inner', op='intersects')

    basin_id_column_name = 'HYBAS_ID'

    for common_area_column in common_area_columns:
        print(f'Processing {common_area_column}')
        gsw_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'gsw_missing_{common_area_column}'].sum()
        my_missing_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'my_missing_{common_area_column}'].sum()
        gsw_missing_area = gsw_missing_area.rename(f'gsw_missing_area_{common_area_column}')
        my_missing_area = my_missing_area.rename(f'my_missing_area_{common_area_column}')
        lake_total_boundary_area = hydrobasins_with_lakes_gdf.groupby(basin_id_column_name)[f'lake_total_boundary_area_{common_area_column}'].sum()
        hydrobasins_gdf = hydrobasins_gdf.merge(gsw_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(my_missing_area, left_on=basin_id_column_name, right_index=True, how='left')
        hydrobasins_gdf = hydrobasins_gdf.merge(lake_total_boundary_area, left_on=basin_id_column_name, right_index=True, how='left')
    
    del hydrobasins_with_lakes_gdf

hydrobasins_gdf.to_pickle(hydrobasins_gdf_with_missing_data_count_and_area_next_quarter_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


Processing 2019-06-01
Processing 2019-07-01
Processing 2019-08-01
Processing 2019-09-01
Processing 2019-10-01
Processing 2019-11-01
Processing 2019-12-01
Processing 2020-01-01
Processing 2020-02-01
Processing 2020-03-01
Processing 2020-04-01
Processing 2020-05-01
Processing 2020-06-01
Processing 2020-07-01
Processing 2020-08-01
Processing 2020-09-01
Processing 2020-10-01
Processing 2020-11-01
Processing 2020-12-01
Processing 2021-01-01
Processing 2021-02-01
Processing 2021-03-01
Processing 2021-04-01
Processing 2021-05-01
Processing 2021-06-01
Processing 2021-07-01
Processing 2021-08-01
Processing 2021-09-01
Processing 2021-10-01
Processing 2021-11-01
Processing 2021-12-01


In [2]:
hydrobasins_gdf

,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,...,lake_total_boundary_area_2021-09-01,gsw_missing_area_2021-10-01,my_missing_area_2021-10-01,lake_total_boundary_area_2021-10-01,gsw_missing_area_2021-11-01,my_missing_area_2021-11-01,lake_total_boundary_area_2021-11-01,gsw_missing_area_2021-12-01,my_missing_area_2021-12-01,lake_total_boundary_area_2021-12-01
0,1020000010,0,1020000010,1020000010,0.0,0.0,3258330.6,3258330.6,11,0,...,3.510877e+10,4.829157e+08,6.250000e+06,3.510877e+10,9.706050e+07,6.000000e+06,3.510877e+10,3.136626e+08,2.975000e+07,3.510877e+10
1,1020011530,0,1020011530,1020011530,0.0,0.0,4660080.9,4660080.9,12,0,...,7.773622e+10,9.870822e+08,1.492500e+08,7.773622e+10,7.680735e+08,1.427500e+08,7.773622e+10,2.682684e+09,4.350000e+07,7.773622e+10
2,1020018110,0,1020018110,1020018110,0.0,0.0,4900405.1,4900405.1,13,0,...,7.738023e+10,1.028588e+10,2.862250e+09,7.738023e+10,7.432942e+09,1.626250e+09,7.738023e+10,1.569738e+10,1.701500e+09,7.738023e+10
3,1020021940,0,1020021940,1020021940,0.0,0.0,4046600.5,4046600.5,14,0,...,4.493237e+10,4.763336e+09,3.955000e+08,4.493237e+10,1.754089e+09,5.675000e+07,4.493237e+10,1.318694e+09,3.950000e+07,4.493237e+10
4,1020027430,0,1020027430,1020027430,0.0,0.0,6923559.6,6923559.6,15,0,...,7.442880e+09,6.466860e+07,2.775000e+07,7.442880e+09,2.358017e+09,2.325000e+07,7.442880e+09,4.521786e+09,2.325000e+07,7.442880e+09
5,1020034170,0,1020034170,1020034170,0.0,0.0,3095083.4,3095083.4,17,0,...,1.240286e+11,7.440461e+09,3.400000e+08,1.240286e+11,2.882060e+10,4.157500e+08,1.240286e+11,4.258519e+09,4.740000e+08,1.240286e+11
6,1020035180,0,1020035180,1020035180,0.0,0.0,597982.7,597982.7,18,0,...,4.184875e+09,1.932876e+08,0.000000e+00,4.184875e+09,1.823211e+08,0.000000e+00,4.184875e+09,2.471850e+07,0.000000e+00,4.184875e+09
7,1020040190,0,1020040190,1020040190,0.0,0.0,2471042.1,2471042.1,16,2,...,1.982411e+10,3.770460e+07,0.000000e+00,1.982411e+10,1.422000e+05,0.000000e+00,1.982411e+10,7.953408e+08,0.000000e+00,1.982411e+10
0,8020000010,0,8020000010,8020000010,0.0,0.0,1782085.5,1782085.5,81,0,...,9.206185e+10,8.939552e+10,1.184250e+09,9.206185e+10,8.939552e+10,8.025000e+08,9.206185e+10,8.939552e+10,1.824600e+10,9.206185e+10
1,8020008900,0,8020008900,8020008900,0.0,0.0,1990718.5,1990718.5,82,0,...,3.516165e+11,3.444666e+11,5.185000e+08,3.516165e+11,3.444666e+11,4.284000e+09,3.516165e+11,3.444666e+11,1.349775e+10,3.516165e+11
